In [1]:
import pandas as pd
import numpy as np

print("🚀 SalesNova Data Preprocessing")
print("=" * 45)

🚀 SalesNova Data Preprocessing


In [2]:
train = pd.read_csv("../data/raw/train.csv")
store = pd.read_csv("../data/raw/store.csv")

print("Train:", train.shape)
print("Store:", store.shape)

Train: (1017209, 9)
Store: (1115, 10)


C:\Users\Aaron Kuriyan\AppData\Local\Temp\ipykernel_22156\3044827146.py:1: DtypeWarning: Columns (0: StateHoliday) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv("../data/raw/train.csv")


In [3]:
train["Date"] = pd.to_datetime(train["Date"])

print(train["Date"].dtype)

datetime64[us]


In [4]:
df = train.merge(
    store,
    on="Store",
    how="left"
)

print("Merged dataset shape:", df.shape)

Merged dataset shape: (1017209, 18)


In [5]:
df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [6]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

Promo2SinceYear              508031
Promo2SinceWeek              508031
PromoInterval                508031
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
CompetitionDistance            2642
dtype: int64


In [7]:
df["HasCompetition"] = df["CompetitionDistance"].notna().astype(int)

In [8]:
df["CompetitionDistance"] = df["CompetitionDistance"].fillna(0)

In [9]:
df["CompetitionOpenSinceMonth"] = (
    df["CompetitionOpenSinceMonth"].fillna(0)
)

df["CompetitionOpenSinceYear"] = (
    df["CompetitionOpenSinceYear"].fillna(0)
)

In [10]:
df["Promo2SinceWeek"] = df["Promo2SinceWeek"].fillna(0)

df["Promo2SinceYear"] = df["Promo2SinceYear"].fillna(0)

df["PromoInterval"] = df["PromoInterval"].fillna("None")

In [11]:
missing = df.isnull().sum()

missing = missing[missing > 0].sort_values(ascending=False)

print(missing)

Series([], dtype: int64)


In [12]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["Day"] = df["Date"].dt.day
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["DayOfWeek"] = df["Date"].dt.dayofweek + 1
df["Quarter"] = df["Date"].dt.quarter

In [13]:
df["IsWeekend"] = (
    df["DayOfWeek"].isin([6, 7])
).astype(int)

In [14]:
df["IsHoliday"] = (
    (df["StateHoliday"] != "0") |
    (df["SchoolHoliday"] == 1)
).astype(int)

In [15]:
print(df["Open"].value_counts())

Open
1    844392
0    172817
Name: count, dtype: int64


In [16]:
model_df = df[df["Open"] == 1].copy()

print("Model dataset:", model_df.shape)

Model dataset: (844392, 26)


In [17]:
model_df.info()

<class 'pandas.DataFrame'>
Index: 844392 entries, 0 to 1017190
Data columns (total 26 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   Store                      844392 non-null  int64         
 1   DayOfWeek                  844392 non-null  int32         
 2   Date                       844392 non-null  datetime64[us]
 3   Sales                      844392 non-null  int64         
 4   Customers                  844392 non-null  int64         
 5   Open                       844392 non-null  int64         
 6   Promo                      844392 non-null  int64         
 7   StateHoliday               844392 non-null  object        
 8   SchoolHoliday              844392 non-null  int64         
 9   StoreType                  844392 non-null  str           
 10  Assortment                 844392 non-null  str           
 11  CompetitionDistance        844392 non-null  float64       
 12  Com

In [18]:
model_df.describe()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,SchoolHoliday,CompetitionDistance,CompetitionOpenSinceMonth,...,Promo2SinceWeek,Promo2SinceYear,HasCompetition,Year,Month,Day,WeekOfYear,Quarter,IsWeekend,IsHoliday
count,844392.000000,844392.000000,844392,844392.000000,844392.000000,844392.0,844392.000000,844392.000000,844392.000000,844392.000000,...,844392.000000,844392.000000,844392.000000,844392.000000,844392.000000,844392.000000,844392.000000,844392.000000,844392.000000,844392.00000
mean,558.422920,3.520361,2014-04-11 01:02:42.487564,6955.514291,762.728395,1.0,0.446352,0.193580,5443.849764,4.926491,...,11.596118,1003.230065,0.997411,2013.831937,5.845738,15.835683,23.646801,2.295995,0.174861,0.29241
min,1.000000,1.000000,2013-01-01 00:00:00,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,2013.000000,1.000000,1.000000,1.000000,1.000000,0.000000,0.00000
25%,280.000000,2.000000,2013-08-16 00:00:00,4859.000000,519.000000,1.0,0.000000,0.000000,700.000000,0.000000,...,0.000000,0.000000,1.000000,2013.000000,3.000000,8.000000,11.000000,1.000000,0.000000,0.00000
50%,558.000000,3.000000,2014-03-31 00:00:00,6369.000000,676.000000,1.0,0.000000,0.000000,2320.000000,4.000000,...,0.000000,0.000000,1.000000,2014.000000,6.000000,16.000000,23.000000,2.000000,0.000000,0.00000
75%,837.000000,5.000000,2014-12-10 00:00:00,8360.000000,893.000000,1.0,1.000000,0.000000,6880.000000,9.000000,...,22.000000,2012.000000,1.000000,2014.000000,8.000000,23.000000,35.000000,3.000000,0.000000,1.00000
max,1115.000000,7.000000,2015-07-31 00:00:00,41551.000000,7388.000000,1.0,1.000000,1.000000,75860.000000,12.000000,...,50.000000,2015.000000,1.000000,2015.000000,12.000000,31.000000,52.000000,4.000000,1.000000,1.00000
std,321.731914,1.723689,NaN,3104.214680,401.227674,0.0,0.497114,0.395103,7804.251737,4.283663,...,15.307873,1005.874806,0.050815,0.777260,3.323931,8.683456,14.389785,1.083483,0.379848,0.45487


In [19]:
model_df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Promo2SinceYear,PromoInterval,HasCompetition,Year,Month,Day,WeekOfYear,Quarter,IsWeekend,IsHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,0.0,None,1,2015,7,31,31,3,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,2010.0,"Jan,Apr,Jul,Oct",1,2015,7,31,31,3,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,2011.0,"Jan,Apr,Jul,Oct",1,2015,7,31,31,3,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,...,0.0,None,1,2015,7,31,31,3,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1,a,...,0.0,None,1,2015,7,31,31,3,0,1


✅ Processed dataset saved successfully!


In [ ]:
model_df.to_csv(
    "../data/processed/sales_processed.csv",
    index=False
)

print("✅ Processed dataset saved successfully!")